# DINOv2 민감도 분석 (2026-10-15)

**목적:** 근거 없이 정한 값들을 바꿔도 결과가 유지되는지 빠른 모델(DINOv2 + Logistic Regression)로 확인한다.
값 하나를 바꿀 때 나머지는 DINOv2 격자(10/13)와 똑같이 둔다.

| 절 | 바꾸는 값 | 범위 | 기존 값 | 다시 학습? |
|---|---|---|---|---|
| S1 | 판정 기준 (확률) | 0.2 ~ 0.8 | 0.5 | 아니오 (점수 재사용) |
| S2 | 질감 게이트 기준 (학습·판정 둘 다) | 5 / 10 / 15 / 20 / 30% | 10% | 예 |
| S3 | buffer (학습·시험 사이 간격) | 0 / 10 / 30 / 60 / 100프레임 | 30 | 예 |
| S4 | 학습에 쓰는 언더필 사진 비율 | 100 / 50 / 25 / 10 / 5% (무작위 3회) | 100% | 예 |
| S5 | Logistic Regression 규제 C | 0.01 / 0.1 / 1 / 10 | 1 | 예 |

**왜 이 값들인가**
- S2·S3은 측정 근거 없이 정한 값이라 논문에서 가장 먼저 질문받을 부분이다.
- S4는 "불량 사진이 적을 때 성능이 얼마나 떨어지나" 곡선이다. 생성 이미지 실험의 출발점(실제 불량만 쓴 기준선)이 된다.
- S1·S5는 판정 기준과 모델 설정이 우연히 좋은 값이었는지 확인한다.

**평가:** 모든 절에서 같은 공통 사진 3,039장(세 선별 모두 통과)으로 비교한다. 같은 사진이라야 값에 따른 차이를 직접 비교할 수 있다. S2는 자기 선별 사진(판정 비율·최장 연속 보류)도 함께 본다.

**조합:** ConvNeXt로 확인한 5개 + 기준선 1개 = 없음 × 없음, 재활용 × 재활용, 질감 × 질감, 2단계 × 질감, 2단계 × 베드 지도, 질감 × 베드 지도

**한계:** 빠른 모델 결과다. DINOv2 순위가 ConvNeXt에서 그대로 재현되지 않은 사례가 있으므로(질감 × 베드 지도), 여기서 민감하게 나온 값만 ConvNeXt로 확인한다.

**필요한 것 (Drive `산공종설/`):** `dinov2_grid_bundle_20261013.zip`, `results/dinov2_grid_20261013/` (10/13 격자 결과: feats_*.npy 4개, gate_values_v2.csv, A_common.csv). **모두 이미 있다.**

**런타임:** GPU 필요 없음 → 런타임 유형 **CPU**로 충분 (특징을 다시 뽑지 않음). 전체 수 분 ~ 20분 [추정]. 결과는 `results/dinov2_sensitivity_20261015/`.

## 1. Drive 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
GRID = f"{DRIVE_DIR}/results/dinov2_grid_20261013"          # 10/13 격자 결과 (특징 재사용)
OUT = f"{DRIVE_DIR}/results/dinov2_sensitivity_20261015"
B = "/content/bundle"
REPRO_STRICT = True   # 기준 설정에서 10/13 결과와 오탐·미탐이 한 장이라도 다르면 멈춤
import os, zipfile
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(f"{B}/selections.csv"):
    with zipfile.ZipFile(f"{DRIVE_DIR}/dinov2_grid_bundle_20261013.zip") as z: z.extractall(B)
for f in ("feats_nomask.npy", "feats_bedmap.npy", "feats_texture.npy", "feats_recycle.npy", "gate_values_v2.csv", "A_common.csv"):
    print("OK  " if os.path.exists(f"{GRID}/{f}") else "없음", f)

## 2. 사진 목록·특징 불러오기 (격자와 같은 순서)
특징은 격자에서 저장한 순서(분할표 검증 행을 파일 이름순)와 같아야 하므로 같은 코드로 목록을 만든다.

In [ ]:
import json, time
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
folds = pd.read_csv(f"{B}/folds.csv", encoding="utf-8-sig")
folds["source"] = folds.source.str.replace("\\", "/", regex=False)
imgs = (folds[folds.role == "val"][["source", "label", "label_id", "frame_index", "fold"]]
        .rename(columns={"fold": "test_fold"}).sort_values("source").reset_index(drop=True))
sel = pd.read_csv(f"{B}/selections.csv", encoding="utf-8-sig")
imgs = imgs.merge(sel.drop(columns=["label", "label_id", "frame_index"]), on="source", how="left", validate="1:1")
gv = pd.read_csv(f"{GRID}/gate_values_v2.csv", encoding="utf-8-sig")
imgs = imgs.merge(gv[["source", "new"]].rename(columns={"new": "texture_ratio"}), on="source", how="left", validate="1:1")
assert len(imgs) == 4000 and imgs.texture_ratio.notna().all()
X = {k: np.load(f"{GRID}/feats_{k}.npy") for k in ("nomask", "bedmap", "texture", "recycle")}
assert all(v.shape == (4000, 1536) for v in X.values())
y = imgs.label_id.values; frame = imgs.frame_index.values; test_fold = imgs.test_fold.values
def gate(t): return imgs.texture_ratio.values >= t
SEL = {"없음": np.ones(4000, bool), "2단계": imgs.stage2.values == 1, "재활용": imgs.recycle.values == 1, "질감": gate(0.10)}
COMMON = SEL["2단계"] & SEL["재활용"] & SEL["질감"]
print("공통 사진", COMMON.sum(), "/ 선별 통과", {k: int(v.sum()) for k, v in SEL.items()})
assert COMMON.sum() == 3039

## 3. 공통 함수 + 기준 설정 재현 확인
- `train_idx(k, buffer)`: fold k의 시험 구간은 그대로 두고, 같은 클래스 시험 사진에서 `buffer`프레임 넘게 떨어진 사진만 학습 후보로 쓴다. buffer 30이면 기존 `folds.csv`와 정확히 같다 (로컬에서 5개 fold 모두 확인함).
- `cv_prob(...)`: fold마다 학습 = 학습 후보 ∩ 선별 통과 (∩ 언더필 일부만), 예측 = 시험 구간 전부 → 4,000장 모두 OOF 점수.
- **재현 확인:** 기본 설정(buffer 30, 게이트 10%, C=1, 언더필 100%)으로 격자 12조합 중 6개를 다시 돌려 10/13 결과(A_common.csv)와 오탐·미탐이 같은지 본다. 다르면 이후 결과를 믿을 수 없으므로 멈춘다.

In [ ]:
MNAME = {"nomask": "없음", "bedmap": "베드 지도", "texture": "질감", "recycle": "재활용"}
KEY = [("없음", "nomask"), ("재활용", "recycle"), ("질감", "texture"), ("2단계", "texture"), ("2단계", "bedmap"), ("질감", "bedmap")]
name = lambda s, m: f"{s} × {MNAME[m]}"
def train_idx(k, buffer):
    keep = np.zeros(4000, bool)
    for lab in (0, 1):
        cls = y == lab; val = frame[cls & (test_fold == k)]
        cand = cls & (test_fold != k)
        d = np.abs(frame[cand][:, None] - val[None, :]).min(1)
        keep[np.where(cand)[0][d > buffer]] = True
    return np.where(keep)[0]
def cv_prob(selmask, m, buffer=30, C=1.0, uf_frac=1.0, seed=0):
    p = np.full(4000, np.nan); ntr = []; rng = np.random.default_rng(seed)
    for k in range(5):
        itr = train_idx(k, buffer); itr = itr[selmask[itr]]
        if uf_frac < 1:
            uf = itr[y[itr] == 1]; keep_uf = rng.choice(uf, max(1, int(round(len(uf) * uf_frac))), replace=False)
            itr = np.sort(np.concatenate([itr[y[itr] == 0], keep_uf]))
        ite = np.where(test_fold == k)[0]
        sc = StandardScaler().fit(X[m][itr])
        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(X[m][itr]), y[itr])
        p[ite] = clf.predict_proba(sc.transform(X[m][ite]))[:, 1]; ntr.append(len(itr))
    assert not np.isnan(p).any()
    return p, ntr
def metrics(p, mask=COMMON, t=0.5):
    yy, pp = y[mask], p[mask]; pred = (pp >= t).astype(int)
    fp = int(((pred == 1) & (yy == 0)).sum()); fn = int(((pred == 0) & (yy == 1)).sum())
    return dict(n=int(mask.sum()), 오탐=fp, 미탐=fn, 틀린수=fp + fn, AUROC=round(roc_auc_score(yy, pp), 4))
def longest_hold(passed):
    best = 0
    for lab in (0, 1):
        idx = np.where(y == lab)[0]; cur = 0
        for v in passed[idx[np.argsort(frame[idx])]]:
            cur = 0 if v else cur + 1; best = max(best, cur)
    return best
# 재현 확인
ref = pd.read_csv(f"{GRID}/A_common.csv", encoding="utf-8-sig").set_index("조합")
BASE, rows = {}, []
for s, m in KEY:
    BASE[(s, m)], ntr = cv_prob(SEL[s], m)
    r = metrics(BASE[(s, m)]); g = ref.loc[name(s, m)]
    same = (r["오탐"], r["미탐"]) == (int(g["오탐"]), int(g["미탐"]))
    rows.append(dict(조합=name(s, m), **r, 격자_오탐=int(g["오탐"]), 격자_미탐=int(g["미탐"]), 일치=same))
R0 = pd.DataFrame(rows); print(R0.to_string(index=False))
R0.to_csv(f"{OUT}/S0_reproduction.csv", index=False, encoding="utf-8-sig")
if REPRO_STRICT: assert R0.일치.all(), "기본 설정 재현 실패: 이후 결과를 쓰지 말고 알려 주세요"

## S1. 판정 기준 (다시 학습 없음)
기본 설정 점수에서 기준만 바꾼다. 오탐과 미탐이 어느 방향으로 얼마나 움직이는지 본다. AUROC는 기준과 무관해서 생략.

In [ ]:
S1 = []
for s, m in KEY:
    for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
        r = metrics(BASE[(s, m)], t=t); S1.append(dict(조합=name(s, m), 기준=t, 오탐=r["오탐"], 미탐=r["미탐"], 틀린수=r["틀린수"]))
S1 = pd.DataFrame(S1); S1.to_csv(f"{OUT}/S1_threshold.csv", index=False, encoding="utf-8-sig")
print(S1.pivot(index="조합", columns="기준", values="틀린수").to_string())

## S2. 질감 게이트 기준 (학습·판정 둘 다 바뀜)
- 학습: 출력물 비율 ≥ 기준인 사진만. 판정: 같은 기준을 통과한 사진만 판정, 나머지는 보류.
- **공통 3,039장** 결과는 기준과 무관하게 같은 사진이라 학습 데이터 변화의 효과만 본다 (기준이 10%보다 높으면 공통 사진 중 일부는 자기 선별 밖이지만 점수는 있음).
- **자기 선별** 결과는 운영 관점: 판정 비율이 줄면 성능이 좋아 보이기 쉬우므로 판정 비율·최장 연속 보류를 같이 본다.

In [ ]:
S2 = []
for t in (0.05, 0.10, 0.15, 0.20, 0.30):
    g = gate(t)
    for m in ("texture", "bedmap"):
        p, ntr = cv_prob(g, m)
        own = metrics(p, mask=g)
        S2.append(dict(마스크=MNAME[m], 게이트기준=t, 학습장수=sum(ntr), **{f"공통_{k}": v for k, v in metrics(p).items() if k != "n"},
                       판정비율=round(g.mean(), 4), 최장연속보류=longest_hold(g), 자기선별_n=own["n"], 자기선별_오탐=own["오탐"],
                       자기선별_미탐=own["미탐"], 자기선별_AUROC=own["AUROC"]))
        print(S2[-1])
S2 = pd.DataFrame(S2); S2.to_csv(f"{OUT}/S2_texture_gate.csv", index=False, encoding="utf-8-sig")

## S3. buffer (학습 구간과 시험 구간 사이 간격)
시험 사진은 그대로, 학습 후보만 바뀐다. buffer가 작을수록 시험 구간 바로 옆 프레임이 학습에 들어간다.
- buffer를 줄일수록 성능이 크게 오르면 → 이웃 프레임 효과(정답 베끼기)가 크다는 뜻. 30에서 안정되면 30이 충분하다는 근거.

In [ ]:
S3 = []
for b in (0, 10, 30, 60, 100):
    for s, m in KEY:
        p, ntr = cv_prob(SEL[s], m, buffer=b)
        S3.append(dict(조합=name(s, m), buffer=b, 학습장수=sum(ntr), **metrics(p)))
    print("buffer", b, "완료")
S3 = pd.DataFrame(S3); S3.to_csv(f"{OUT}/S3_buffer.csv", index=False, encoding="utf-8-sig")
print(S3.pivot(index="조합", columns="buffer", values="틀린수").to_string())
print(S3.pivot(index="조합", columns="buffer", values="AUROC").to_string())

## S4. 학습 언더필 사진 비율 (불량이 적을 때)
fold마다 학습 언더필 사진을 무작위로 일부만 남긴다(정상은 전부). 무작위 뽑기 3회의 평균·표준편차.
- 언더필이 정상보다 훨씬 적어지면 확률 0.5 기준이 정상 쪽으로 치우친다 → **AUROC를 주 지표**로 보고, 오탐·미탐은 참고.
- 무작위 뽑기는 서로 다른 시점의 불량을 고루 갖게 된다. 실제로는 불량이 한두 번의 사건(연속 구간)에서만 나올 수 있어 이보다 불리할 수 있다 [추정].

In [ ]:
S4 = []
for frac in (1.0, 0.5, 0.25, 0.10, 0.05):
    for s, m in [("재활용", "recycle"), ("질감", "texture"), ("2단계", "texture")]:
        for rep in (range(1) if frac == 1.0 else range(3)):
            p, ntr = cv_prob(SEL[s], m, uf_frac=frac, seed=rep)
            S4.append(dict(조합=name(s, m), 언더필비율=frac, 반복=rep, 학습장수=sum(ntr), **metrics(p)))
    print("언더필 비율", frac, "완료")
S4 = pd.DataFrame(S4); S4.to_csv(f"{OUT}/S4_underfill_fraction.csv", index=False, encoding="utf-8-sig")
agg = S4.groupby(["조합", "언더필비율"]).agg(AUROC평균=("AUROC", "mean"), AUROC_sd=("AUROC", "std"),
                                             오탐평균=("오탐", "mean"), 미탐평균=("미탐", "mean"), 학습장수=("학습장수", "mean")).round(4)
print(agg.to_string())

## S5. Logistic Regression 규제 C

In [ ]:
S5 = []
for C in (0.01, 0.1, 1.0, 10.0):
    for s, m in KEY:
        p, _ = cv_prob(SEL[s], m, C=C)
        S5.append(dict(조합=name(s, m), C=C, **metrics(p)))
    print("C", C, "완료")
S5 = pd.DataFrame(S5); S5.to_csv(f"{OUT}/S5_regularization.csv", index=False, encoding="utf-8-sig")
print(S5.pivot(index="조합", columns="C", values="틀린수").to_string())

## 결과 묶기 + 런타임 반납
`results/dinov2_sensitivity_20261015.zip`을 받아서 주면 된다.

In [ ]:
import shutil
json.dump(dict(base=dict(buffer=30, gate=0.10, C=1.0, uf_frac=1.0, threshold=0.5), combos=[name(*k) for k in KEY],
               common=int(COMMON.sum()), features="results/dinov2_grid_20261013/feats_*.npy (vit_base_patch14_dinov2.lvd142m, CLS+patch mean)"),
          open(f"{OUT}/run_info.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(shutil.make_archive(f"{DRIVE_DIR}/results/dinov2_sensitivity_20261015", "zip", OUT))

In [ ]:
from google.colab import runtime
runtime.unassign()